# Turn a diagram; count its layers
### Young diagrams, conjugate partitions, and measured packing

The heights $(5,3,2)$ describe ten cells. A different list, $(3,3,2,1,1)$,
describes those same cells from another direction. What information survives this change?

We will derive the second list by counting, use it to build a fresh arrangement,
turn the original cells through 3D space, and use cumulative measurements to pack
them into a strip. A final counterexample shows why counts do not preserve order.

[Lesson notes](../docs/lessons/06_young_layers.md) · [Setup](README.md)

In [ ]:
from pathlib import Path
from math import pi, sqrt
import json
import sys
import numpy as np
import plotly.io as pio
from IPython.display import Markdown, display
from kaleion import Collection, F, Inspection, Ref, Motion, Workspace, cos, sin, param

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").exists() and (p / "src/kaleion").is_dir())
sys.path.insert(0, str(ROOT / "notebooks"))
from lesson_views import COLORS, cell_panels, profiles, replay, save_figures
pio.renderers.default = "plotly_mimetype+notebook"
OUTPUT = ROOT / "build/notebooks/young-layers"
OUTPUT.mkdir(parents=True, exist_ok=True)

HEIGHTS = [5, 3, 2, 0]  # The explicit empty column is retained in the measurements.
assert all(isinstance(h, int) and not isinstance(h, bool) and h >= 0 for h in HEIGHTS)
assert HEIGHTS == sorted(HEIGHTS, reverse=True), "Use decreasing heights here; an unordered case follows."
N, BOUND = len(HEIGHTS), max([1] + HEIGHTS)
assert N <= 12 and BOUND <= 12, "Use a small fully displayed diagram."

## 1 · Construct cells under a height arrangement

A partition lists positive heights in nonincreasing order. We allow trailing zero
columns in the stored arrangement, while the partition itself omits them.

For column $i$ and layer $\ell\ge1$, select the cell when $\ell\le h_i$.
The code retains zero-based logical indices `i,j`; the displayed `column,level`
coordinates start at 1. Counting by `F.i` retains columns. Counting by `F.j`
retains layers, including layers whose count is zero.

In [ ]:
heights = Collection.literal(HEIGHTS)
grid = (Collection.grid(N, BOUND, values=1)
        .annotate(column=F.i + 1, level=F.j + 1).arrange(F.column, F.level, 0))
diagram = grid.where(F.level <= heights.bind(on=F.i))
columns = diagram.count(by=F.i).annotate(column=F.i + 1)
layers = diagram.count(by=F.j).annotate(level=F.j + 1)

# Counts become the heights of a new, independently identified arrangement.
new_grid = (Collection.grid(BOUND, N, values=1)
            .annotate(column=F.i + 1, level=F.j + 1).arrange(F.column, F.level, 0))
conjugate = new_grid.where(F.level <= layers.bind(on=F.i, key=F.j))
twice = conjugate.count(by=F.j).annotate(column=F.j + 1)

# Give the selected original cells stable layer labels for motion coloring.
original_cells = diagram.select().with_values(F.level)
turned_cells = original_cells.arrange(F.level, F.column, 0)
ROOTS = {"grid": grid, "diagram": diagram, "columns": columns, "layers": layers,
         "conjugate": conjugate, "twice": twice,
         "cells": original_cells, "turned": turned_cells}
workspace = Workspace(ROOTS)
assert not workspace.state.errors, dict(workspace.state.errors)
state = workspace.state
r = state.results
assert r["columns"].values.tolist() == r["twice"].values.tolist() == HEIGHTS
assert sum(r["layers"].values) == sum(HEIGHTS) == r["diagram"].cardinality
original_positions = {tuple(p) for p in r["turned"].positions}
new_positions = {tuple(p) for p in r["conjugate"].source.positions[r["conjugate"].mask]}
assert original_positions == new_positions
assert set(r["cells"].ids).isdisjoint(r["conjugate"].source.ids)
print("Column counts:", r["columns"].values.tolist())
print("Layer counts:", r["layers"].values.tolist())
print("Conjugate twice:", r["twice"].values.tolist())

In [ ]:
diagram_plot = cell_panels([r["diagram"], r["turned"], r["conjugate"]],
    ["Original cells", "Transposed originals", "Rebuilt from counts"],
    title="Same occupied coordinates · two different ways to obtain them")
diagram_plot.show()

## 2 · Let the measurements decide where each row begins

If layer $\ell$ contains $c_\ell$ cells, its starting offset in a packed strip is

\[
o_\ell=\sum_{t<\ell}c_t.
\]

Declare the order of the measured layers and take their **exclusive prefix sums**.
The first offset is zero, and each layer contributes its measured length to later
layers. The core stores a compact ordered contributor range instead of expanding
all layer pairs. No evaluated list is copied back as anonymous data.


In [ ]:
offsets = layers.group_by().order_by(F.j).prefix_sums(key=F.j).annotate(level=F.j + 1)
strip = turned_cells.arrange(F.column + offsets.bind(on=F.j, key=F.j), 1, 0)

# Keep the small dense construction as an independent finite reference, not a driver.
earlier_layers = Collection.grid(BOUND, BOUND, values=1).annotate(
    length=layers.bind(on=F.j, key=F.j))
reference_offsets = earlier_layers.where(F.j < F.i).sum(by=F.i, value=F.length)
np.testing.assert_array_equal(offsets.evaluate().values, reference_offsets.evaluate().values)

workspace.set("offsets", offsets)
workspace.set("strip", strip)
assert not workspace.state.errors, dict(workspace.state.errors)
packed = workspace.state.results["strip"]
assert sorted(map(int, packed.positions[:, 0])) == list(range(1, sum(HEIGHTS) + 1))
assert packed.ids == r["cells"].ids
counts_plot = profiles([r["columns"], r["layers"], workspace.state.results["offsets"]],
    ["Column heights", "Layer counts", "Measured starting offsets"],
    keys=["column", "level", "level"], title="Measurements become arguments for placement")
counts_plot.show()

## 3 · Turn, pack, and undo

Transposing $(x,y)$ is a reflection in the plane. To show it without collapsing
points during a straight interpolation, rotate the plane through $180^\circ$ in 3D
about the line $x=y,z=0$. Then use the measured offsets to pack the cells into a strip.

Colors identify their original layers. Markers represent cell centers; their screen
sizes do not measure area. The exact endpoint coordinates define the arrangements;
the intervening path is presentation. Undo samples that same recorded path backward.

In [ ]:
t = param("time")
flip = Motion.custom(
    (F.sx + F.sy + (F.sx - F.sy) * cos(pi * t)) / 2,
    (F.sx + F.sy - (F.sx - F.sy) * cos(pi * t)) / 2,
    (F.sy - F.sx) * sin(pi * t) / sqrt(2))
motion_workspace = Workspace({"cells": original_cells})
turn = motion_workspace.set("cells", turned_cells, motion=flip)
pack = motion_workspace.set("cells", strip, motion=Motion.arc(height=1.5, axis=2, dimension=3))
motion_workspace.capture("The same cells transpose, then pack using measured layer offsets.")
unpack, unturn = motion_workspace.undo(), motion_workspace.undo()
samples, captions = [], []
for name, transition in (("turn through 3D", turn), ("pack measured layers", pack),
                         ("undo packing", unpack), ("undo the turn", unturn)):
    for progress in np.linspace(0, 1, 21):
        samples.append(transition.frame("cells", float(progress)))
        captions.append(f"{name} · {progress:.0%}")
np.testing.assert_allclose(samples[0].positions, samples[-1].positions, atol=1e-12, rtol=0)
for outward, inward in ((turn, unturn), (pack, unpack)):
    np.testing.assert_array_equal(outward.frame("cells", .25).positions, inward.frame("cells", .75).positions)
colors = {oid: COLORS[(int(value) - 1) % len(COLORS)] for oid, value in zip(r["cells"].ids, r["cells"].values)}
motion_plot = replay(samples, captions, title="Count the layers · turn the cells · pack their measured lengths", colors_by_id=colors)
motion_plot.show()

## 4 · Count the same finite set in two orders

\[
\boxed{\sum_i h_i=\sum_{\ell\ge1}\#\{i:h_i\ge\ell\}.}
\]

Both sides count pairs $(i,\ell)$ with $1\le\ell\le h_i$. Counting first by column
or first by layer changes the grouping, not the number of cells. For a partition,
the occupied columns in every layer form an initial interval. Transposition therefore
gives another partition, its **conjugate**. Transposing twice restores the original.

The prefix sum gives disjoint consecutive intervals of lengths $c_\ell$, so the
packed strip contains exactly the same number of cells. The double-counting identity
holds for every finite nonnegative height list. The packing recipe above uses the
ordered-height assumption: the column number is already a consecutive rank within
each layer. An unordered list needs an explicit within-layer rank before packing.

The [Sage partition reference](https://doc.sagemath.org/html/en/reference/combinat/sage/combinat/partition.html)
describes conjugate partitions and their diagram conventions. We use a column-height
orientation and expose the transpose explicitly. A diagram is not yet a standard
or semistandard Young tableau: those require conditions on a filling.

## 5 · What did the counts forget?

The ordered lists $(5,3,2)$ and $(2,5,3)$ have identical layer counts. Rebuilding
contiguous columns from those counts recovers the decreasing rearrangement, not
the original order. This changes the values assigned to column keys; merely
reordering a driver's storage while retaining its keys would leave its action unchanged.

In [ ]:
counter_grid = Collection.grid(3, 5, values=1).arrange(F.i + 1, F.j + 1)
ordered = counter_grid.where(F.j < Collection.literal([5, 3, 2]).bind(on=F.i))
unordered = counter_grid.where(F.j < Collection.literal([2, 5, 3]).bind(on=F.i))
unordered_layers = unordered.count(by=F.j)
counter_conjugate = Collection.grid(5, 3, values=1).where(
    F.j < unordered_layers.bind(on=F.i, key=F.j))
recovered_heights = counter_conjugate.count(by=F.j)
canonical = counter_grid.where(F.j < recovered_heights.bind(on=F.i, key=F.j))
counter_workspace = Workspace({"ordered": ordered, "unordered": unordered, "canonical": canonical,
                               "counts_A": ordered.count(by=F.j), "counts_B": unordered_layers,
                               "recovered_heights": recovered_heights})
cr = counter_workspace.state.results
assert not counter_workspace.state.errors
assert cr["counts_A"].values.tolist() == cr["counts_B"].values.tolist() == [3, 3, 2, 1, 1]
assert cr["recovered_heights"].values.tolist() == [5, 3, 2]
assert not np.array_equal(cr["unordered"].mask, cr["canonical"].mask)
order_plot = cell_panels([cr["ordered"], cr["unordered"], cr["canonical"]],
    ["Heights (5,3,2)", "Heights (2,5,3)", "Rebuilt from their counts"],
    title="Identical layer measurements can forget the original column order")
order_plot.show()

## 6 · Keep a measured layer's explanation

Choose a layer, follow its count to the original cells, and inspect the prefix offset
used to move it. Empty layers have zero contributors and zero length. The main
construction also accepts the empty height list; the chosen finite layer bound
then retains one zero group rather than invoking a maximum of an empty list.

In [ ]:
LAYER = min(3, BOUND)
source = r["grid"]
by_id = {oid: [int(source.fields["column"][i]), int(source.fields["level"][i])]
         for i, oid in enumerate(source.ids)}
contributors = r["layers"].contributor_ids(LAYER - 1)
explanation = {"layer": LAYER, "count": int(r["layers"].values[LAYER - 1]),
               "offset": int(workspace.state.results["offsets"].values[LAYER - 1]),
               "contributors": [{"occurrence": oid, "cell": by_id[oid]} for oid in contributors]}
# The offset's contributors are measured layers; each has its own cell evidence.
measured_offsets = workspace.state.results["offsets"]
inspector = Inspection(workspace.state)
offset_receipt = inspector.measurement(Ref(measured_offsets.node, measured_offsets.ids[LAYER - 1]))
explanation["offset_evidence"] = offset_receipt.to_dict()
explanation["earlier_layer_evidence"] = [inspector.measurement(c.item.ref).to_dict()
                                          for c in offset_receipt.contributors]
assert sum(c.weight for c in offset_receipt.contributors) == explanation["offset"]
assert all(c.item.value == c.weight for c in offset_receipt.contributors)
assert len(contributors) == explanation["count"]
print("Layer", LAYER, "has", explanation["count"], "cells and begins after", explanation["offset"], "cells.")
print("Original contributors:", [item["cell"] for item in explanation["contributors"]])

workspace.capture("Column/layer double counting, a derived conjugate, and measured packing offsets.")
save_figures(OUTPUT, {"diagrams": diagram_plot, "measurements": counts_plot,
                      "turn-pack-undo": motion_plot, "lost-order": order_plot})
for name, investigation in (("measurements", workspace), ("motion", motion_workspace), ("order", counter_workspace)):
    payload = investigation.to_json()
    (OUTPUT / f"{name}-workspace.json").write_text(payload)
    restored = Workspace.from_json(payload)
    assert not restored.state.errors
restored = Workspace.from_json(workspace.to_json())
assert restored.state.results["layers"].contributor_ids(LAYER - 1) == contributors
assert Inspection(restored.state).measurement(offset_receipt.item.ref).to_dict() == offset_receipt.to_dict()
(OUTPUT / "layer-explanation.json").write_text(json.dumps(explanation, indent=2))
(OUTPUT / "checks.json").write_text(json.dumps({"heights": HEIGHTS,
    "layers": list(map(int, r["layers"].values)), "cells": sum(HEIGHTS),
    "offsets": list(map(int, workspace.state.results["offsets"].values)),
    "motion_frames": len(samples)}, indent=2))
print("Saved four offline figures, captured workspaces, and the layer explanation to", OUTPUT)

The repeated structure is **measure groups → derive cumulative offsets → preserve
occurrences while placing them elsewhere**. Counts alone do not supply an ordering.
Next, [convolution and additive energy](07_additive_structure.ipynb) will need a
related count: how many earlier occurrences share the same sum?